# Pipelines - Revamped Version


ce notebook sert pour tester le backend des pipelines

# Initialisation

In [ ]:
%pip install tqdm

## Imports Librairies

In [ ]:
import os
import sys
from tqdm import tqdm
import glob
import numpy as np
from PIL import Image
from sklearn.base import BaseEstimator, TransformerMixin, ClassifierMixin
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns




"""import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)
"""



## Ajouter le répertoire racine du projet au chemin Python ( à remplacer par toml)

In [ ]:
project_root = os.path.abspath(os.path.join(os.getcwd(), '../..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

## Imports Features et Transformateurs

In [ ]:
"""
from src.features.Pipelines.Transformateurs.image_augmentation import *
from src.features.Pipelines.Transformateurs.image_features import *
from src.features.Pipelines.Transformateurs.image_loaders import *
from src.features.Pipelines.Transformateurs.image_preprocessing import *
from src.features.Pipelines.Transformateurs.utilities import *
from src.features.Pipelines.loading_pipelines import *
from src.features.Pipelines.Transformateurs.keras_classifiers import *
"""

In [ ]:
from src.features.Pipelines.loading_pipelines import *
from src.features.Pipelines.Transformateurs import *

## Checkup Transformateurs (Optionnel)

In [ ]:
# Scanner automatique des transformateurs disponibles
def discover_transformers():
    """Découvre automatiquement tous les transformateurs disponibles"""
    transformateurs_path = os.path.join(project_root, "src", "features", "Pipelines", "Transformateurs")
    transformers_found = {}
    transformers_with_seed = {}
    
    # Scanner tous les fichiers Python dans le dossier Transformateurs
    for file_path in glob.glob(os.path.join(transformateurs_path, "*.py")):
        file_name = os.path.basename(file_path)
        
        # Ignorer __init__.py et __pycache__
        if file_name.startswith("__"):
            continue
            
        module_name = file_name[:-3]  # Retirer .py
        
        try:
            # Importer le module dynamiquement
            module = importlib.import_module(f"src.features.Pipelines.Transformateurs.{module_name}")
            
            # Chercher toutes les classes qui héritent de BaseEstimator
            for name in dir(module):
                obj = getattr(module, name)
                if (inspect.isclass(obj) and 
                    hasattr(obj, 'fit') and 
                    hasattr(obj, 'transform') and
                    name not in ['BaseEstimator', 'TransformerMixin', 'ClassifierMixin']):
                    
                    if module_name not in transformers_found:
                        transformers_found[module_name] = []
                        transformers_with_seed[module_name] = []
                    
                    transformers_found[module_name].append(name)
                    
                    # Vérifier si le transformateur supporte random_state
                    seed_support = check_seed_support(obj)
                    if seed_support['has_seed']:
                        transformers_with_seed[module_name].append({
                            'name': name,
                            'method': seed_support['method']
                        })
                    
        except Exception as e:
            print(f"⚠️  Erreur lors de l'import de {module_name}: {e}")
    
    return transformers_found, transformers_with_seed

def check_seed_support(transformer_class):
    """Vérifie si un transformateur supporte random_state"""
    seed_info = {'has_seed': False, 'method': None}
    
    try:
        # Méthode 1: Vérifier dans __init__
        init_signature = inspect.signature(transformer_class.__init__)
        if 'random_state' in init_signature.parameters:
            seed_info = {'has_seed': True, 'method': 'constructor'}
            return seed_info
            
        # Méthode 2: Vérifier les attributs de classe
        if hasattr(transformer_class, 'random_state'):
            seed_info = {'has_seed': True, 'method': 'attribute'}
            return seed_info
            
        # Méthode 3: Vérifier si c'est un wrapper sklearn
        try:
            # Créer une instance temporaire pour tester
            temp_instance = transformer_class()
            if hasattr(temp_instance, 'set_params'):
                # Tester si set_params accepte random_state
                try:
                    temp_instance.set_params(random_state=42)
                    seed_info = {'has_seed': True, 'method': 'set_params'}
                except (TypeError, ValueError):
                    pass
        except:
            pass  # Ignore si on ne peut pas créer d'instance
            
    except Exception:
        pass
    
    return seed_info

# Découvrir et afficher les transformateurs
import importlib
import inspect

print("🔍 Découverte automatique des transformateurs:")
print("=" * 50)

transformers_by_module, transformers_with_seed_by_module = discover_transformers()

total_transformers = 0
total_with_seed = 0

for module_name, transformer_names in transformers_by_module.items():
    print(f"\n📁 Module: {module_name}")
    print("-" * 30)
    
    # Transformateurs avec seed
    seed_transformers = transformers_with_seed_by_module.get(module_name, [])
    seed_names = [t['name'] for t in seed_transformers]
    
    for transformer_name in transformer_names:
        if transformer_name in seed_names:
            # Trouver la méthode de seed
            method = next(t['method'] for t in seed_transformers if t['name'] == transformer_name)
            print(f"  ✅ {transformer_name} 🎲 (seed via {method})")
            total_with_seed += 1
        else:
            print(f"  ✅ {transformer_name}")
        total_transformers += 1

print(f"\n📊 Résumé:")
print(f"  Total transformateurs: {total_transformers}")
print(f"  Avec support seed: {total_with_seed}")
print(f"  Sans seed: {total_transformers - total_with_seed}")

"""# Affichage détaillé des transformateurs avec seed
print(f"\n🎲 Transformateurs avec support de seed:")
print("=" * 50)

for module_name, seed_transformers in transformers_with_seed_by_module.items():
    if seed_transformers:
        print(f"\n📁 {module_name}:")
        for transformer_info in seed_transformers:
            name = transformer_info['name']
            method = transformer_info['method']
            method_emoji = {
                'constructor': '🏗️',
                'attribute': '📝', 
                'set_params': '⚙️'
            }.get(method, '❓')
            print(f"  🎲 {name} {method_emoji} ({method})")"""

print(f"\n📁 Chemin du projet ajouté: {project_root}")
print(f"📂 Répertoire de travail actuel: {os.getcwd()}")

## Configs

### Variables Environnement

In [ ]:
root_dir = '../../data/raw/COVID-19_Radiography_Dataset/COVID-19_Radiography_Dataset/'

In [ ]:
seed = 42

# Main

## Chargement Paths

In [ ]:
X, masks, y = load_paths_data_raw(root_dir)

In [ ]:
print(f"Nombre d'images: {len(X)}")
print(f"Labels uniques: {set(y)}")

## Split des jeux de données

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed)
print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")

## Chargement des pipelines disponibles

In [ ]:
# Découverte automatique des pipelines disponibles
registry_config = discover_available_pipelines(os.path.join(project_root, "src/features/Pipelines/Configs_Pipelines/"))
display_pipeline_structure(registry_config)

## Séléction Pipeline

In [ ]:
# Chargement du pipeline par défaut avec seed
default_pipeline_name = "feature_engineering"

default_config = load_pipeline_config(registry_config[default_pipeline_name]["file"])
pipeline = create_pipeline_from_config(default_config, masks)

print(f"\nPipeline par défaut chargé:\n  {default_config['name']}")
print(f"\nDescription:\n  {default_config['description']}")
print(f"\nNombre d'étapes:\n  {len(pipeline.steps)}")

for step_name, step in pipeline.steps:
    print(f"  - Étape: {step_name} | Type: {type(step).__name__}")

# Appliquer la seed
set_pipeline_random_state(pipeline, seed)

In [ ]:
pipeline

## Training Pipeline

In [ ]:
# Entraînement du pipeline composite
pipeline.fit(X_train, y_train)

## Prédiction Pipeline

In [ ]:
# Prédiction
y_pred = pipeline.predict(X_test)

## Matrice De Confusion

In [ ]:
# Matrice de confusion visuelle
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=set(y_test), yticklabels=set(y_test))
plt.xlabel('Prédit')
plt.ylabel('Vrai')
plt.title('Matrice de confusion')
plt.show()



## Quelques exemples d'images (Vrai vs. Prédit)

In [ ]:
# Afficher quelques images test avec prédiction et vrai label
nbr_images = 3
for i in range(nbr_images):
    img = Image.open(X_test[i])
    plt.imshow(img, cmap='gray')
    plt.title(f"Vrai: {y_test[i]} / Prédit: {y_pred[i]}")
    plt.axis('off')
    plt.show()